# Cached on-policy analysis

This notebook is an analysis-only attachment. It reads existing hidden-state, logprob, and report caches; it does not load a model or make a forward pass. All joins use validated item IDs, and all ranking risks use the shared corrected AURC implementation.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import torch

HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / 'experiments' / 'ladder.py').exists())
PAPER = ROOT / 'verbalized_conformal_confidence'
sys.path.insert(0, str(ROOT))
from experiments.ladder import _split
from scripts.step_token_contribution import AURC_CONVENTION, aurc, build_targets, load_cell
from utils.paths import logprobs_path
print({'root': str(ROOT), 'aurc_convention': AURC_CONVENTION})

## 1. Cache inventory and provenance

`example_ids` is accepted as legacy provenance only after hidden-state and logprob arrays match exactly. No positional fallback is permitted.

In [ ]:
cells = []
for hp in sorted((ROOT / 'outputs' / 'step1_extract').glob('*/*/hidden_states.pt')):
    model, dataset = hp.parent.parent.name, hp.parent.name
    hs = torch.load(hp, map_location='cpu', weights_only=False)
    lp_path = hp.with_name('logprobs.pt')
    lp = torch.load(lp_path, map_location='cpu', weights_only=False) if lp_path.exists() else {}
    hs_ids = hs.get('row_id', hs.get('example_ids'))
    lp_ids = lp.get('row_id', lp.get('example_ids'))
    ids_match = hs_ids is not None and lp_ids is not None and np.array_equal(np.asarray(hs_ids).astype(str), np.asarray(lp_ids).astype(str))
    cells.append({'model': model, 'dataset': dataset, 'n': len(hs.get('h_q', hs.get('h_pos', []))), 'has_h_q': 'h_q' in hs, 'ids_match': ids_match})
inventory = pd.DataFrame(cells)
inventory[inventory.has_h_q].sort_values(['model', 'dataset'])

## 2. ID-based cache loader

The returned frame is keyed by `row_id`; variants are aligned by an inner ID join rather than assumed row order.

In [ ]:
def cache_frame(model, dataset):
    d, err = load_cell(model, dataset)
    if err: raise RuntimeError(err['error'])
    y = build_targets(d)['mean'][0]
    return pd.DataFrame({'row_id': d['row_id'].astype(str), 'y': y,
                         'h_q_available': True,
                         'lp_pos_mean': d['lp_pos_mean'],
                         'lp_neg_mean': d['lp_neg_mean']})

def id_join(left, right, suffixes=('_left', '_right')):
    if left.row_id.duplicated().any() or right.row_id.duplicated().any():
        raise ValueError('duplicate item IDs')
    out = left.merge(right, on='row_id', how='inner', suffixes=suffixes, validate='one_to_one')
    if out.empty: raise ValueError('empty ID join')
    return out

## 3. Updated E1 predictions and splits

These are the exported cached predictions from the corrected ladder run. Seeds and the original 40% test split are retained.

In [ ]:
e1_json = ROOT / 'ladder_e1_updated.json'
e1 = json.loads(e1_json.read_text())
e1_rows = []
for cell, result in e1.items():
    r = result.get('E1_gap', {})
    if 'mean' in r:
        e1_rows.append({'cell': cell, 'n_questions': r['n_questions'], 'base_rate': r['base_rate'],
                        'AURC_q': r['mean']['AURC_q'], 'AURC_a': r['mean']['AURC_a'],
                        'gap': r['mean']['gap'], 'gap_sd': r['sd']['gap'],
                        'aurc_convention': r['aurc_convention']})
e1_summary = pd.DataFrame(e1_rows)
e1_summary

## 4. Report-channel rescoring

The report score is the cached `V+ - V-` readout. This cell only displays the recomputed robustness table; it does not reinterpret the target as candidate correctness.

In [ ]:
report_dir = PAPER / 'cached_results' / 'report_robustness'
report_summary = pd.read_csv(report_dir / 'probe_readout_summary.csv')
report_summary[report_summary['readout'].eq('fixed_mean_gap')]

## 5. Split and prediction export audit

The audit checks that every exported row has a stable ID, that train/test membership is explicit, and that no item is assigned to both partitions within a seed.

In [ ]:
export_root = ROOT / 'outputs' / 'p2_exports'
split_files = sorted(export_root.glob('*/*/split_assignments.csv'))
prediction_files = sorted(export_root.glob('*/*/heldout_predictions.csv'))
export_audit = []
for path in split_files:
    x = pd.read_csv(path, dtype={'row_id': str})
    dup = x.duplicated(['seed', 'row_id']).sum()
    export_audit.append({'file': str(path.relative_to(ROOT)), 'rows': len(x), 'duplicate_seed_ids': int(dup), 'seeds': sorted(x.seed.unique().tolist())})
pd.DataFrame(export_audit)

## 6. Tie convention

All reported AURC values use the shared discrete expected-uniform-within-ties convention. Exact score equality defines a tie; no rounded values are used to create or break ties.

In [ ]:
print(AURC_CONVENTION)
print('test fraction = 0.4; seeds = (0, 1, 2); cache mode = True')

## 7. Interpretation boundary

E1 compares question-only and answer-side state probes. The report tables are protocol-conditioned outcomes under the cached verbalization; legend reversal and format sensitivity remain measurement findings, not causal mechanism claims.

In [ ]:
legend = pd.read_csv(PAPER / 'cached_results' / 'legend_stability' / 'summary.csv')
legend.head()

## Cell 11 — diagnostics only

This cell reports cache-health diagnostics only. It does not fit probes, compute calibration metrics, make model calls, or add a scientific result.

In [ ]:
diagnostics = inventory.assign(
    duplicate_or_missing_ids=~inventory.ids_match,
    usable_for_e1=inventory.has_h_q & inventory.ids_match,
)
diagnostics[['model', 'dataset', 'n', 'has_h_q', 'ids_match', 'duplicate_or_missing_ids', 'usable_for_e1']]